In [ ]:
# this example defined by ./examples/validation/reference_projection.py
#  defines a random coefficient vector c and uses dubiner bases to 
#  create a function interior to the span of the dubiner bases
#  then we project it into the basis space by u = M^{-1}f

import numpy as np
from examples.validation.reference_projection import ReferenceProjectionTest
import numpy as np

p=5
dim = int((1/2)*(p+1)*(p+2))
rng = np.random.default_rng(200)
c = rng.normal(size=dim)

# c = np.zeros(dim)
# c = normal.rng()

test = ReferenceProjectionTest(p=p, c=c)

XI, ETA = np.meshgrid(
    np.linspace(-1, 1, 50),
    np.linspace(-1, 1, 50)
)

dump = test.run()

In [ ]:
# Here we load the symbolic modified dubiner basis functions of order p=3
# then we take the triangle_to_square map defined in reference_triangle.py,
# it is used to map reference space (r,s) variables to warped dubiner space (xi, eta)
# where the basis functions can be evaluated.  
# a sanity check lets you see the symbolic represenation vs the evaluation at a given point (r,s)

from localtfem.modified_dubiner.dubiner_sym import dubiner_basis, dubiner_basis_lambdified
from localtfem.geometry.reference_triangle import triangle_to_square
import sympy as sp
import numpy as np

xi, eta = sp.symbols("xi eta")

basis_sym = dubiner_basis(3)
basis = dubiner_basis_lambdified(3)
print( basis_sym )
xi, eta = triangle_to_square( 0, 0 )
[basis[i]( xi, eta ) for i in range(len(basis))]


In [ ]:
# building off the last example, here we plot with contours dubiner 
# basis of order p<=3

import numpy as np
import matplotlib.pyplot as plt
from localtfem.geometry.reference_triangle import square_to_triangle
from localtfem.modified_dubiner.dubiner_sym import dubiner_basis_lambdified
import math

# ------------------------------------------------------------
# basis
# ------------------------------------------------------------
p=3
basis = dubiner_basis_lambdified(p)
n_basis = len(basis)

# ------------------------------------------------------------
# grid in square space
# ------------------------------------------------------------
n = 50
xi_vals = np.linspace(-1, 1, n)
eta_vals = np.linspace(-1, 1, n)

XI, ETA = np.meshgrid(xi_vals, eta_vals)

# ------------------------------------------------------------
# map to triangle
# ------------------------------------------------------------
R, S = square_to_triangle(XI, ETA)

# ------------------------------------------------------------
# evaluate all basis functions
# ------------------------------------------------------------
Phi = np.stack([phi(XI, ETA) for phi in basis], axis=-1)

# ------------------------------------------------------------
# plot
# ------------------------------------------------------------

titles = [r"$\phi_A$", r"$\phi_B$", r"$\phi_C$", 
          r"$\phi_{A1}$", r"$\phi_{B1}$", r"$\phi_{C1}$", 
          r"$\phi_{A2}$", r"$\phi_{B2}$", r"$\phi_{C2}$", 
          r"$\phi_{I1}$" ]
ncols = 3

nrows = math.ceil(n_basis / ncols)

fig, axs = plt.subplots(
    nrows,
    ncols,
    figsize=(4*ncols, 4*nrows),
    constrained_layout=True,
)

# Flatten so we can index linearly
axs = np.atleast_1d(axs).ravel()

for i in range(n_basis):
    ax = axs[i]

    # Filled contours
    cf = ax.contourf(R, S, Phi[..., i], levels=30, cmap="viridis")

    # Three contour lines
    cs = ax.contour(
        R, S, Phi[..., i],
        levels=3,
        colors="black",
        linewidths=1.0,
    )
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

    ax.set_title(titles[i])
    ax.set_aspect("equal")
    plt.colorbar(cf, ax=ax, shrink=0.65)

# Hide any unused axes
for ax in axs[n_basis:]:
    ax.axis("off")

plt.show()
plt.tight_layout()
plt.show()